In [28]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import pandas as pd

# Load the training and test datasets
train_df = pd.read_csv(r"C:\Projects\HackX2026\PlugMind\Source\Datasets\Loan_Prediction_Train.csv")
test_df = pd.read_csv(r"C:\Projects\HackX2026\PlugMind\Source\Datasets\Loan_Prediction_Test.csv")

# Show data preview
print("Training Data Sample:")
print(train_df.head())

Training Data Sample:
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0             360.0   
3             2583             2358.0       120.0             360.0   
4             6000                0.0       141.0             360.0   

   Credit_History Property_Area Loan_Status  
0             1.0         Urban           Y  
1             1.0         Rural           N 

In [30]:
# Check shape of the dataset
print("Dataset shape:", train_df.shape)

# List all column names
print("Column names:", train_df.columns.tolist())

# Check for missing values
print("Missing values:\n", train_df.isnull().sum())

# Summary statistics of numerical columns
train_df.describe()

Dataset shape: (614, 13)
Column names: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']
Missing values:
 Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [38]:
# Drop rows with missing values in critical columns
train_df = train_df.dropna()
test_df = test_df.dropna()

# Confirm missing values are handled
print("Missing values after cleaning:\n", train_df.isnull().sum())
print("\nTraining data shape after cleaning:", train_df.shape)
print("Test data shape after cleaning:", test_df.shape)

Missing values after cleaning:
 Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

Training data shape after cleaning: (529, 13)
Test data shape after cleaning: (328, 12)


In [32]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
from sklearn.preprocessing import LabelEncoder

# Create label encoder objects
le_dict = {}

# List of categorical columns
categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 
                    'Self_Employed', 'Property_Area', 'Loan_Status']

# Apply label encoding to each column
for col in categorical_cols:
    le_dict[col] = LabelEncoder()
    train_df[col] = le_dict[col].fit_transform(train_df[col].astype(str))
    # Apply same encoding to test data if column exists
    if col in test_df.columns and col != 'Loan_Status':
        test_df[col] = le_dict[col].transform(test_df[col].astype(str))

train_df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
1,LP001003,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,0
2,LP001005,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,2,1
3,LP001006,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,2,1
4,LP001008,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,2,1
5,LP001011,1,1,2,0,1,5417,4196.0,267.0,360.0,1.0,2,1


In [40]:
from sklearn.model_selection import train_test_split

# Select input features (X) and target variable (y)
X = train_df.drop(['Loan_ID', 'Loan_Status'], axis=1)  # All columns except ID and target
y = train_df['Loan_Status']               # Target column

# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Print shape to verify
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Training set shape: (423, 11)
Test set shape: (106, 11)


Model 1: Logistic Regression

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Train logistic regression model
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# Predict on test set
log_pred = log_model.predict(X_test)

# Calculate accuracy
log_acc = accuracy_score(y_test, log_pred)

# Show results
print("Logistic Regression Results:")
print("Predictions:", log_pred)
print("Accuracy:", round(log_acc * 100, 2), "%")

Logistic Regression Results:
Predictions: [0 0 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 0 1 0 1 1 0
 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 0 1]
Accuracy: 83.02 %


c:\Users\Tavis Fernandes\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model 2: Decision Tree Classifier

In [42]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Train the Decision Tree model
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

# Predict on test data
tree_preds = tree_model.predict(X_test)

# Calculate accuracy
tree_acc = accuracy_score(y_test, tree_preds)

# Show results
print("Decision Tree Classifier Results:")
print("Predictions:", tree_preds)
print("Accuracy:", round(tree_acc * 100, 2), "%\n")

Decision Tree Classifier Results:
Predictions: [0 0 0 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 0 1 1 0 1 1 1 0 0 1 0 1 0 0 0 1 0 0
 1 0 0 1 0 1 0 1 1 1 0 0 1 0 1 1 0 1 0 1 1 1 1 0 0 1 1 0 1 1 1 1 1 0 1 0 1
 1 1 1 1 0 1 1 1 1 0 0 1 1 1 0 1 0 0 1 1 0 1 1 1 1 1 1 1 1 1 0 1]
Accuracy: 69.81 %



Model 3: Random Forest Classifier

In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Train random forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test set
rf_pred = rf_model.predict(X_test)

# Calculate accuracy
rf_acc = accuracy_score(y_test, rf_pred)

# Print results
print("Random Forest Classifier Results:")
print("Predictions:", rf_pred)
print("Accuracy:", round(rf_acc * 100, 2), "%")

Random Forest Classifier Results:
Predictions: [0 0 0 1 1 1 1 1 1 1 1 0 1 0 1 1 1 0 1 1 1 1 0 1 1 1 1 0 1 0 1 0 1 0 1 1 0
 1 1 1 1 0 1 1 1 1 1 0 1 1 0 1 1 0 1 0 1 0 0 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 0 1]
Accuracy: 80.19 %


Model 4: Support Vector Machine

In [44]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Train SVM model
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train, y_train)

# Predict on test set
svm_pred = svm_model.predict(X_test)

# Calculate accuracy
svm_acc = accuracy_score(y_test, svm_pred)

# Print results
print("Support Vector Machine (SVM) Results:")
print("Predictions:", svm_pred)
print("Accuracy:", round(svm_acc * 100, 2), "%")

Support Vector Machine (SVM) Results:
Predictions: [0 0 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 0
 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1]
Accuracy: 80.19 %


Model Evaluation

In [45]:
# Display comparison of all models
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(f"Logistic Regression Accuracy: {round(log_acc * 100, 2)}%")
print(f"Decision Tree Accuracy: {round(tree_acc * 100, 2)}%")
print(f"Random Forest Accuracy: {round(rf_acc * 100, 2)}%")
print(f"SVM Accuracy: {round(svm_acc * 100, 2)}%")
print("="*50)


MODEL COMPARISON
Logistic Regression Accuracy: 83.02%
Decision Tree Accuracy: 69.81%
Random Forest Accuracy: 80.19%
SVM Accuracy: 80.19%


In [46]:
import joblib

# Export all trained models
joblib.dump(log_model, 'logistic_regression_model.pkl')
joblib.dump(tree_model, 'decision_tree_model.pkl')
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(svm_model, 'svm_model.pkl')

# Export label encoders
joblib.dump(le_dict, 'label_encoders.pkl')

print("All models and encoders exported successfully!")
print("Files saved:")
print("- logistic_regression_model.pkl")
print("- decision_tree_model.pkl")
print("- random_forest_model.pkl")
print("- svm_model.pkl")
print("- label_encoders.pkl")

All models and encoders exported successfully!
Files saved:
- logistic_regression_model.pkl
- decision_tree_model.pkl
- random_forest_model.pkl
- svm_model.pkl
- label_encoders.pkl
